# 4. Customising SimCityNet

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hpa-code/simcitynet/blob/main/docs/tutorials/04_customising.ipynb)

*Run it in Colab (free GPU) to see every figure; the website shows the saved outputs.*

This notebook shows how to run without a GPU, change settings, plug in your own model, choose
a normalisation, and use the command line.

In [ ]:
# Installs simcitynet with all models (PyTorch, Transformers, Ultralytics YOLO).
# In Colab, choose Runtime > Change runtime type > GPU for speed.
%pip install -q "simcitynet[all]"
# Before the first PyPI release, install from GitHub instead:
# %pip install -q "simcitynet[all] @ git+https://github.com/hpa-code/simcitynet.git"

In [ ]:
from simcitynet import Config, SimCityNet
from simcitynet.datasets import fetch_examples

paths = fetch_examples("examples")

## 1. Colour only — no deep learning, runs anywhere

The colour block needs no neural network, so it works with `pip install simcitynet` alone.
Blocks you skip are left empty (`NaN`) and ignored in comparisons.

In [ ]:
fast = SimCityNet(blocks=["color"])
colours = fast.embed(paths)
colours.block("color").round(1).iloc[:, 15:]     # the five shares

In [ ]:
colours.plot.palette("doha")

In [ ]:
colours.distance("color_emd").round(1)

## 2. Change settings

All settings live in `Config` (see *Faithfulness to the paper* for every default).

In [ ]:
cfg = Config(
    detection_conf=0.4,       # stricter detection threshold (default 0.25)
    depth_weighting=False,    # plain Green View Index instead of depth-weighted
    max_side=1024,            # analyse smaller images (faster)
)
cfg

Shortcut: keyword arguments to `SimCityNet` override single settings.

In [ ]:
scn = SimCityNet(detection_conf=0.4, device="cuda")

## 3. Plug in your own model

Any object with the right method can replace a model. Here a tiny vegetation segmenter based on
the *excess green* index (2G − R − B) replaces Mask2Former — no download, runs on CPU.

In [ ]:
import numpy as np

class ExcessGreenSegmenter:
    """Labels clearly green pixels as 'tree' (class 1); everything else is class 0."""

    labels = {0: "other", 1: "tree"}

    def segment(self, image):
        rgb = image.astype(int)
        exg = 2 * rgb[..., 1] - rgb[..., 0] - rgb[..., 2]
        return (exg > 40).astype(np.int32)

light = SimCityNet(
    Config(depth_weighting=False),
    blocks=["vegetation", "color"],
    vegetation_segmenter=ExcessGreenSegmenter(),
)
green = light.embed(paths)
green.raw["vegetation"].round(3)

The same works for `depth_estimator=`, `surface_segmenter=` and `detector=` (see the
`simcitynet.backends` protocols in the API reference). This is also how to avoid the AGPL
licence of Ultralytics: pass a detector with a licence that suits you.

## 4. Normalisation

`"batch"` (default) scales each dimension by its maximum in the current set, as in the paper.
`"fixed"` uses fixed bounds, so results stay comparable when you embed more images later.

In [ ]:
full = SimCityNet().embed(paths)
print("batch:\n", full.similarity(normalize="batch").round(3), "\n")
print("fixed:\n", full.similarity(normalize="fixed").round(3))

## 5. Save, load and the command line

In [ ]:
from simcitynet import Embeddings

full.save("results/six_cities")
again = Embeddings.load("results/six_cities.csv")
again.similarity().round(3)

In [ ]:
!simcitynet features | head -8
!simcitynet similarity results/six_cities.csv -o results/six_cities_emd.csv --metric color_emd
!simcitynet doctor